# Exoplanet Transit Candidate Vetting Prototype  
## A notebook-based first version built around real Kepler light curves

I want to start by being precise about what I am building here, because the wording matters.

This notebook is not a full end-to-end exoplanet discovery pipeline. That would be too ambitious and, more importantly, it would be the wrong claim for the kind of data product I am using. What I am building here is a **transit candidate vetting prototype**. I start from Kepler objects that are already known to be transit-like enough to appear in archive tables, I retrieve their light curves, I search those light curves for their strongest repeating box-shaped dip, I engineer features from the recovered signal, and I train a first classifier that tries to separate planet-like cases from non-planetary transit-like cases.

I prefer this framing because it is more honest and more professional. It does not weaken the project. It strengthens it. The moment I describe the task precisely, every later decision becomes easier to defend: the label source, the use of BLS, the choice of engineered features, the evaluation strategy, and the limitations.

I also want this notebook to be something I can actually study from. For that reason I am explaining every step carefully. I am not treating any part of the workflow as self-explanatory. If a concept matters for the project, I want it to appear in the notebook in a way that I can understand, remember, and later explain out loud without sounding like I am reciting words I never really digested.

## Why I reframed the task as vetting instead of detection

The Kepler archive distinguishes between several levels of signal processing. A **Threshold-Crossing Event**, usually shortened to **TCE**, is a sequence of transit-like features in a light curve that is strong enough to be passed onward for further analysis. A **Kepler Object of Interest**, or **KOI**, is further downstream: it is a transit-like astrophysical object that has already gone through more scrutiny. NASA’s archive overview explicitly describes KOIs as TCEs that, after further analysis, are judged to be astrophysical transiting or eclipsing objects. That detail is not cosmetic. It means that if I build a classifier on KOIs, I am already working in a post-search, post-screening regime. I am not starting from arbitrary stars. I am starting from objects that have already been promoted into the candidate-analysis workflow. citeturn673795search4turn673795search10turn459586search5

That is why the cleanest claim for this notebook is **candidate vetting**. I am taking transit-like archive objects and asking whether the signal extracted from the light curve behaves more like a confirmed planet case or more like a false-positive case. That is a serious and realistic prototype to show in six days. It proves that the project already has a technically meaningful backbone without pretending that I have rebuilt the full Kepler search pipeline from raw spacecraft pixels. citeturn673795search10turn673795search12

## Official references I am building on

I want the notebook to stand on professional sources, not on random blog logic. These are the main references behind the workflow.

The Lightkurve exoplanet tutorial explains how to identify transiting signals in Kepler and TESS light curves and explicitly states that it uses Astropy’s Box Least Squares implementation for transit searches. The Lightkurve tutorial index also shows that exoplanet recovery, machine-learning style preprocessing, and interactive BLS exploration are all part of the intended analysis ecosystem. The Astropy BLS documentation explains that Box Least Squares is commonly used to detect transiting exoplanets and eclipsing binaries in photometric time-series data. citeturn821495search3turn821495search1turn673795search1turn673795search3

For the labels and data products, I rely on the NASA Exoplanet Archive documentation. The archive’s data overview explains the relationship between TCEs and KOIs. The KOI column documentation defines archive dispositions such as `CONFIRMED` and `FALSE POSITIVE`. The Certified False Positive documentation describes that table as a high-reliability set examined by the False Positive Working Group and makes an important distinction between false positives and false alarms. The False Positive Probabilities table description is also useful background because it documents an automated FPP calculation for Kepler transit signals. citeturn673795search10turn459586search10turn673795search0turn673795search2

For noise-aware feature engineering, I rely on Lightkurve’s `estimate_cdpp()` documentation. It explains that CDPP is a transit-detectability noise metric and that the Lightkurve implementation uses a Savitzky-Golay proxy approach after removing long-term trends. That makes it a natural feature in a transit-vetting model because a dip is only meaningful relative to the noise floor around it. citeturn821495search0

## Environment note before I run anything

This notebook uses `lightkurve`, `astropy`, `scikit-learn`, and Jupyter widgets. In practice, the astronomy stack behaves more smoothly on Python 3.11, 3.12, or 3.13 than on Python 3.14. If I run into package issues, I should create a clean environment on Python 3.12 and use that as the notebook kernel.

I also keep the installation cell visible. I do that on purpose. A prototype becomes more credible when the environment assumptions are explicit instead of hidden.

In [ ]:
# Run this once in a fresh environment if needed.

# %pip install -q lightkurve pandas numpy matplotlib scikit-learn requests ipywidgets bokeh

In [ ]:
from pathlib import Path
from io import StringIO
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import lightkurve as lk

from astropy.timeseries import BoxLeastSquares

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_auc_score
)

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Project settings

I am surfacing the key settings near the top because I want the notebook to be easy to steer during a demo.

The sample size per class is deliberately modest. This is a first prototype, and I would rather have a smaller set of targets that I process carefully than a larger set that I only half understand. The period range is also restricted. Long-period signals are harder to recover quickly because there are fewer observed transit events in a finite time baseline. For a first prototype, it is smarter to work in a regime where repeated events are easier to recover and easier to explain.

I am also caching intermediate results. That does not make the notebook less scientific. It makes it demonstrable. The first run can do the heavy work. Later runs can reuse the same downloaded products and feature tables so that I can focus on explanation instead of waiting for repeated archive calls.

In [ ]:
RANDOM_STATE = 42

N_CONFIRMED = 18
N_NEGATIVE = 18

MIN_PERIOD_DAYS = 0.75
MAX_PERIOD_DAYS = 20.0

PERIOD_GRID_SIZE = 2500
DURATION_GRID_HOURS = np.arange(1, 13, 1)
PHASE_BINS = 160

REBUILD_FEATURES = True
CFP_TABLE_OVERRIDE = None
FPP_TABLE_OVERRIDE = None

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
LIGHTKURVE_CACHE_DIR = DATA_DIR / "lightkurve_cache"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, LIGHTKURVE_CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_CACHE_PATH = PROCESSED_DIR / "selected_targets_v2.csv"
FEATURE_CACHE_PATH = PROCESSED_DIR / "transit_features_v2.csv"
FAILURE_CACHE_PATH = PROCESSED_DIR / "failed_targets_v2.csv"
CV_RESULTS_PATH = PROCESSED_DIR / "cross_validation_results_v2.csv"
FEATURE_IMPORTANCE_PATH = PROCESSED_DIR / "feature_importance_v2.csv"
PREDICTIONS_PATH = PROCESSED_DIR / "holdout_predictions_v2.csv"

## Utility functions for archive access

The NASA Exoplanet Archive exposes a TAP service. TAP stands for Table Access Protocol. I am using it because it lets me retrieve both data rows and schema metadata programmatically. This matters for a professional notebook because it lets me inspect what tables exist, what columns they contain, and whether the strongest label source is directly available through the API I am using.

I do not want to hardcode assumptions blindly. When possible, I want the notebook to inspect the archive and tell me what it found.

In [ ]:
def run_tap_query(query: str, timeout: int = 180) -> pd.DataFrame:
    url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    response = requests.get(url, params={"query": query, "format": "csv"}, timeout=timeout)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))


def fetch_tap_schema_tables() -> pd.DataFrame:
    return run_tap_query("select schema_name, table_name, description from TAP_SCHEMA.tables")


def describe_tap_table(table_name: str) -> pd.DataFrame:
    query = f"""
    select table_name, column_name, datatype, description
    from TAP_SCHEMA.columns
    where table_name = '{table_name}'
    order by column_name
    """
    return run_tap_query(query)


def first_existing_column(columns_df: pd.DataFrame, preferred_names):
    available = {c.lower(): c for c in columns_df["column_name"].astype(str)}
    for name in preferred_names:
        if name.lower() in available:
            return available[name.lower()]
    return None

In [ ]:
tap_tables_df = fetch_tap_schema_tables()
interesting_tables_df = tap_tables_df[
    tap_tables_df["table_name"].fillna("").str.contains("cumulative|cert|cfp|fpp|tce", case=False, regex=True)
    |
    tap_tables_df["description"].fillna("").str.contains("certified false positive|false positive|threshold-crossing|koi", case=False, regex=True)
].sort_values("table_name")

interesting_tables_df.head(30)

## Reading the schema discovery step

I like this step because it turns the notebook into something more self-aware. Instead of silently assuming that every useful Kepler table is available in the exact same interface, I ask the archive what it exposes through TAP and then inspect the answer.

This is useful for two reasons. First, it helps me locate stronger label sources such as a Certified False Positive table if it is available through the same service. Second, it makes the fallback logic easier to justify. If I end up using a fallback negative class, I can show that I tried the stronger route first and only dropped back when the API path did not cooperate.

In [ ]:
CUMULATIVE_TABLE = "cumulative"

def resolve_table_name(tables_df: pd.DataFrame, override: str, preferred_patterns):
    if override is not None:
        return override

    local = tables_df.copy()
    local["combined_text"] = (
        local["table_name"].fillna("").astype(str) + " " +
        local["description"].fillna("").astype(str)
    ).str.lower()

    for pattern in preferred_patterns:
        matches = local[local["combined_text"].str.contains(pattern, regex=True, na=False)]
        if len(matches) > 0:
            return matches.iloc[0]["table_name"]
    return None

CFP_TABLE = resolve_table_name(
    interesting_tables_df,
    CFP_TABLE_OVERRIDE,
    preferred_patterns=[r"certified.*false.*positive", r"false.*positive.*cert", r"\bcfp\b", r"cert"]
)

FPP_TABLE = resolve_table_name(
    interesting_tables_df,
    FPP_TABLE_OVERRIDE,
    preferred_patterns=[r"false.*positive.*prob", r"\bfpp\b"]
)

print("Resolved cumulative table:", CUMULATIVE_TABLE)
print("Resolved certified false-positive table:", CFP_TABLE)
print("Resolved FPP table:", FPP_TABLE)

In [ ]:
cumulative_columns_df = describe_tap_table(CUMULATIVE_TABLE)
cumulative_columns_df.head(40)

In [ ]:
if CFP_TABLE is not None:
    cfp_columns_df = describe_tap_table(CFP_TABLE)
    cfp_columns_df.head(60)
else:
    cfp_columns_df = pd.DataFrame()
    print("No CFP table was resolved through TAP. The notebook will fall back to archive false positives if needed.")

## Label design strategy

This is one of the most important design choices in the whole notebook.

For the positive class, I use objects from the cumulative KOI table whose archive disposition is `CONFIRMED`. That part is straightforward.

For the negative class, I prefer a stronger source than the raw `FALSE POSITIVE` slice of the cumulative KOI table. The strongest option for this notebook is to identify objects that also appear in the archive’s **Certified False Positive** table, because that table is explicitly described by NASA as a high-reliability set examined by the False Positive Working Group. That gives the negative class a cleaner interpretation. These are not merely “not confirmed.” They are cases that have stronger evidence against a planetary explanation. citeturn673795search0

At the same time, I need the notebook to remain executable. So I implement a professional fallback. I first try to resolve and use Certified False Positives through TAP. If that path does not work cleanly, I fall back to the cumulative KOI table’s `FALSE POSITIVE` disposition and make that fallback explicit in the notebook. I would rather be transparent than silently pretend that the stronger label source was used when it was not.

In [ ]:
def fetch_cumulative_koi_table() -> pd.DataFrame:
    query = f"""
    select
        kepid,
        kepoi_name,
        koi_disposition,
        koi_period,
        koi_time0bk,
        koi_duration,
        koi_depth,
        koi_prad,
        koi_impact
    from {CUMULATIVE_TABLE}
    where koi_disposition in ('CONFIRMED', 'FALSE POSITIVE')
      and koi_period between {MIN_PERIOD_DAYS} and {MAX_PERIOD_DAYS}
      and koi_duration is not null
      and koi_depth is not null
      and kepid is not null
    """
    df = run_tap_query(query)
    df = (
        df.sort_values(["kepid", "koi_depth"], ascending=[True, False])
          .drop_duplicates(subset=["kepid"], keep="first")
          .reset_index(drop=True)
    )
    return df


def fetch_certified_false_positive_keys(cfp_table: str, cfp_columns_df: pd.DataFrame):
    if cfp_table is None or len(cfp_columns_df) == 0:
        return None, None

    possible_cfp_cols = ["kepoi_name", "kepid", "koi_name", "kepler_name", "kic", "kic_id"]
    chosen_key = first_existing_column(cfp_columns_df, possible_cfp_cols)
    if chosen_key is None:
        return None, None

    query = f"select distinct {chosen_key} as cfp_key from {cfp_table} where {chosen_key} is not null"
    keys_df = run_tap_query(query)
    return chosen_key, keys_df


def choose_negative_class(cumulative_df: pd.DataFrame, cfp_table: str, cfp_columns_df: pd.DataFrame):
    cfp_key_name, cfp_keys_df = fetch_certified_false_positive_keys(cfp_table, cfp_columns_df)

    if cfp_key_name is not None and cfp_keys_df is not None and len(cfp_keys_df) > 0:
        cfp_keys = set(cfp_keys_df["cfp_key"].astype(str))

        if cfp_key_name.lower() == "kepoi_name" and "kepoi_name" in cumulative_df.columns:
            negatives = cumulative_df[cumulative_df["kepoi_name"].astype(str).isin(cfp_keys)].copy()
            if len(negatives) > 0:
                negatives["negative_label_source"] = "Certified False Positive table"
                return negatives, "Certified False Positive table", cfp_key_name

        if cfp_key_name.lower() == "kepid" and "kepid" in cumulative_df.columns:
            negatives = cumulative_df[cumulative_df["kepid"].astype(str).isin(cfp_keys)].copy()
            if len(negatives) > 0:
                negatives["negative_label_source"] = "Certified False Positive table"
                return negatives, "Certified False Positive table", cfp_key_name

    negatives = cumulative_df[cumulative_df["koi_disposition"] == "FALSE POSITIVE"].copy()
    negatives["negative_label_source"] = "KOI cumulative FALSE POSITIVE fallback"
    return negatives, "KOI cumulative FALSE POSITIVE fallback", None

In [ ]:
cumulative_df = fetch_cumulative_koi_table()
negative_pool_df, negative_strategy_used, negative_match_key = choose_negative_class(cumulative_df, CFP_TABLE, cfp_columns_df)

confirmed_pool_df = cumulative_df[cumulative_df["koi_disposition"] == "CONFIRMED"].copy()
confirmed_pool_df["negative_label_source"] = np.nan

print("Confirmed pool size:", len(confirmed_pool_df))
print("Negative pool size:", len(negative_pool_df))
print("Negative strategy used:", negative_strategy_used)
print("Negative matching key:", negative_match_key)

In [ ]:
def build_balanced_target_set(confirmed_df: pd.DataFrame, negative_df: pd.DataFrame) -> pd.DataFrame:
    n = min(N_CONFIRMED, N_NEGATIVE, len(confirmed_df), len(negative_df))

    confirmed_sample = confirmed_df.sample(n=n, random_state=RANDOM_STATE).copy()
    negative_sample = negative_df.sample(n=n, random_state=RANDOM_STATE).copy()

    confirmed_sample["label"] = 1
    negative_sample["label"] = 0

    targets = (
        pd.concat([confirmed_sample, negative_sample], ignore_index=True)
          .sample(frac=1.0, random_state=RANDOM_STATE)
          .reset_index(drop=True)
    )
    return targets

targets_df = build_balanced_target_set(confirmed_pool_df, negative_pool_df)
targets_df.to_csv(TARGET_CACHE_PATH, index=False)

print(f"Final prototype sample size: {len(targets_df)}")
targets_df.head(10)

In [ ]:
targets_df.groupby(["label", "koi_disposition"]).size()

## What the target table means

At this point I have a balanced classification dataset at the object level, but I still do not have model-ready samples.

Each row tells me which target I am going to process. It includes the Kepler target identifier, the KOI name, the archive disposition, and a few archive-side parameters such as period, duration, and depth. I am not using those archive solution parameters as the final model features. I keep them for two different reasons.

The first reason is quality control. If my BLS recovery finds a period that is wildly inconsistent with the archive period for many objects, that is a sign that something in my preprocessing or period search setup needs attention.

The second reason is intellectual honesty. The point of this notebook is to classify based on features I extract from the light curves themselves. If I fed the archive’s own curated transit parameters directly into the model, I would be blurring the line between the raw observational signal and the archive’s downstream characterization work. For a clean prototype, I want the features to come from my own analysis of the light curve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
targets_df["label"].map({1: "Confirmed", 0: "Negative"}).value_counts().plot(kind="bar", ax=ax)
ax.set_title("Balanced prototype sample")
ax.set_xlabel("Class")
ax.set_ylabel("Number of targets")
plt.show()

## From archive rows to usable light curves

A row in a catalog is not yet a scientific signal. To reach the signal, I still need to retrieve the actual brightness measurements for the star.

Lightkurve is useful here because it already knows how to search mission archives, download preprocessed light-curve products, and represent them as objects that can be cleaned, normalized, stitched, flattened, folded, and inspected. Its documentation also shows that `stitch()` can apply a correction function before combining multiple segments, which is exactly what I want when a Kepler target has several observing quarters. I do not want to naively glue together raw segments with different baselines. I want each segment normalized first so that the combined curve is meaningful. citeturn821495search5turn821495search6turn821495search12

The preparation sequence I use is deliberate. I remove NaNs because missing measurements are not informative to the periodogram. I normalize because transit depth is interpreted relative to the local baseline flux. I remove large outliers because extreme spikes can distort a transit search. Then I flatten the light curve to suppress long-timescale trends and preserve the short-duration structure that is most relevant for transits. Lightkurve’s flattening operation uses a Savitzky-Golay filter for this purpose. citeturn821495search4turn821495search2turn821495search0

In [ ]:
def choose_window_length(n_points: int) -> int:
    candidate = max(101, min(901, n_points // 20))
    if candidate % 2 == 0:
        candidate += 1
    return candidate


def download_and_prepare_lightcurve(kepid: int):
    search = lk.search_lightcurve(
        f"KIC {int(kepid)}",
        mission="Kepler",
        author="Kepler",
        cadence="long",
    )

    if len(search) == 0:
        raise ValueError("No Kepler light-curve products were found for this target.")

    collection = search.download_all(download_dir=str(LIGHTKURVE_CACHE_DIR))
    if collection is None or len(collection) == 0:
        raise ValueError("Archive search succeeded, but no downloadable light curves were returned.")

    stitched = collection.stitch(corrector_func=lambda lc: lc.remove_nans().normalize())
    stitched = stitched.remove_nans().remove_outliers(sigma=5)

    window_length = choose_window_length(len(stitched.time))
    flattened = stitched.flatten(window_length=window_length, polyorder=2, break_tolerance=5)
    flattened = flattened.remove_nans()

    if len(flattened.time) < 500:
        raise ValueError("Too few cadences remain after cleaning and flattening.")

    return stitched, flattened

## Why Box Least Squares is the right search tool here

I need to pause here because this is one of the core scientific ideas in the notebook.

A transit is not shaped like a sine wave. The star stays near its baseline for most of the orbit, then there is a short drop, then the flux returns. If I used a method optimized for smooth sinusoidal oscillations, I would be asking the wrong question. I would be trying to explain a box-like phenomenon with a wave-like model.

Box Least Squares solves the right problem. It searches many trial periods and durations and asks which repeating box-shaped dimming event best matches the data. Astropy’s documentation is explicit that BLS is commonly used for discovering transiting exoplanets and eclipsing binaries in photometric time-series data. Lightkurve’s own exoplanet tutorial is built around this same approach. That makes BLS the natural signal-search engine for this prototype. citeturn673795search1turn673795search3turn821495search3

I also like BLS for a first classifier because it produces interpretable outputs. The best period, best duration, best depth, best transit epoch, best power, and related statistics are not abstract latent values. They describe the strongest recovered transit-like candidate in a way I can inspect, plot, and later explain.

In [ ]:
def robust_flux_arrays(lightcurve):
    time = np.asarray(lightcurve.time.value, dtype=float)
    flux = np.asarray(getattr(lightcurve.flux, "value", lightcurve.flux), dtype=float)

    if getattr(lightcurve, "flux_err", None) is not None:
        flux_err = np.asarray(getattr(lightcurve.flux_err, "value", lightcurve.flux_err), dtype=float)
        if not np.isfinite(flux_err).all() or np.nanmedian(flux_err) <= 0:
            flux_err = np.full_like(flux, np.nanstd(flux))
    else:
        flux_err = np.full_like(flux, np.nanstd(flux))

    mask = np.isfinite(time) & np.isfinite(flux) & np.isfinite(flux_err)
    time = time[mask]
    flux = flux[mask]
    flux_err = flux_err[mask]

    if len(time) < 500:
        raise ValueError("Too few usable observations remain after finite-value filtering.")

    return time, flux, flux_err


def mad_normal(x):
    median = np.nanmedian(x)
    return 1.4826 * np.nanmedian(np.abs(x - median))


def build_period_duration_grids():
    periods = np.linspace(MIN_PERIOD_DAYS, MAX_PERIOD_DAYS, PERIOD_GRID_SIZE)
    durations = DURATION_GRID_HOURS / 24.0
    return periods, durations


def fold_arrays(time, flux, period, t0):
    phase = ((time - t0 + 0.5 * period) % period) / period - 0.5
    order = np.argsort(phase)
    return phase[order], flux[order]


def phase01_arrays(time, flux, period, t0):
    phase01 = ((time - t0) % period) / period
    order = np.argsort(phase01)
    return phase01[order], flux[order]


def binned_phase_curve(phase, flux, n_bins=PHASE_BINS, low=-0.5, high=0.5):
    bins = np.linspace(low, high, n_bins + 1)
    mids = 0.5 * (bins[:-1] + bins[1:])
    idx = np.digitize(phase, bins) - 1
    binned = np.array([
        np.nanmedian(flux[idx == i]) if np.any(idx == i) else np.nan
        for i in range(n_bins)
    ])
    return mids, binned


def circular_distance(phase01, center):
    d = np.abs(phase01 - center)
    return np.minimum(d, 1 - d)


def event_depth_from_phase01(phase01, flux, center=0.0, half_width=0.03, exclusion_width=0.08):
    target_mask = circular_distance(phase01, center) <= half_width
    baseline_mask = (
        (circular_distance(phase01, center) > exclusion_width) &
        (circular_distance(phase01, 0.5) > exclusion_width)
    )

    if target_mask.sum() < 3 or baseline_mask.sum() < 10:
        return np.nan

    baseline = np.nanmedian(flux[baseline_mask])
    event_flux = np.nanmedian(flux[target_mask])
    return baseline - event_flux


def compute_odd_even_depth_difference(time, flux, period, duration, t0):
    centers = []
    k_min = int(np.floor((time.min() - t0) / period)) - 1
    k_max = int(np.ceil((time.max() - t0) / period)) + 1

    for k in range(k_min, k_max + 1):
        centers.append(t0 + k * period)

    depths = []
    for center in centers:
        in_mask = np.abs(time - center) <= (duration / 2)
        local_mask = np.abs(time - center) <= (2 * duration)
        oot_mask = local_mask & (~in_mask)

        if in_mask.sum() >= 3 and oot_mask.sum() >= 5:
            baseline = np.nanmedian(flux[oot_mask])
            event = np.nanmedian(flux[in_mask])
            depths.append(baseline - event)

    if len(depths) < 4:
        return np.nan, len(depths)

    odd_depth = np.nanmedian(depths[::2])
    even_depth = np.nanmedian(depths[1::2])
    return np.abs(odd_depth - even_depth), len(depths)


def compute_phase_symmetry_metric(phase, flux, window=0.08, n_bins=40):
    mask = np.abs(phase) <= window
    if mask.sum() < 20:
        return np.nan

    bins = np.linspace(-window, window, n_bins + 1)
    idx = np.digitize(phase[mask], bins) - 1
    binned = np.array([
        np.nanmedian(flux[mask][idx == i]) if np.any(idx == i) else np.nan
        for i in range(n_bins)
    ])

    half = n_bins // 2
    left = binned[:half]
    right = binned[half:][::-1]
    valid = np.isfinite(left) & np.isfinite(right)

    if valid.sum() < 5:
        return np.nan

    return np.nanmean(np.abs(left[valid] - right[valid]))


def compute_transit_sharpness(phase, flux, inner=0.01, shoulder_low=0.02, shoulder_high=0.05):
    inner_mask = np.abs(phase) <= inner
    shoulder_mask = (np.abs(phase) >= shoulder_low) & (np.abs(phase) <= shoulder_high)

    if inner_mask.sum() < 3 or shoulder_mask.sum() < 3:
        return np.nan

    center_flux = np.nanmedian(flux[inner_mask])
    shoulder_flux = np.nanmedian(flux[shoulder_mask])
    return shoulder_flux - center_flux

## Feature engineering: what I am trying to capture and why

This part is the bridge between the astrophysical signal and the machine learning model.

I do not want to throw the entire light curve directly into the classifier at this stage. That would make the model harder to interpret and harder to defend in a first prototype. Instead, I want each target to become a compact numerical description of the strongest transit-like signal I recovered.

Some features summarize the candidate itself. These include the BLS best period, duration, depth, and power. They describe what the search thinks the strongest repeating dip looks like.

Some features summarize the background noise. These include the flux standard deviation, the median-absolute-deviation scale, and the CDPP proxies. These matter because a given transit depth means something different in a quiet light curve than in a noisy one.

Some features summarize morphology. The odd-even depth difference matters because eclipsing binaries often produce alternating events with different depths. The secondary-event proxy near phase 0.5 matters because a secondary eclipse can be a warning sign against a planetary interpretation. The phase symmetry metric and the transit sharpness metric try to capture whether the folded event looks like a clean central dip rather than a messy or asymmetric shape.

I like this feature family because every column has a reason to exist. I can justify each one as part of a planet-versus-non-planet story.

In [ ]:
def extract_transit_features(flattened_lc, catalog_period=None):
    time, flux, flux_err = robust_flux_arrays(flattened_lc)
    periods, durations = build_period_duration_grids()

    bls = BoxLeastSquares(time, flux, dy=flux_err)
    results = bls.power(periods, durations, objective="snr")

    powers = np.asarray(results.power, dtype=float)
    best_idx = int(np.nanargmax(powers))

    best_period = float(np.asarray(results.period)[best_idx])
    best_duration = float(np.asarray(results.duration)[best_idx])
    best_t0 = float(np.asarray(results.transit_time)[best_idx])
    best_depth = float(np.asarray(results.depth)[best_idx])
    best_power = float(np.asarray(results.power)[best_idx])
    best_depth_snr = float(np.asarray(results.depth_snr)[best_idx])

    transit_mask = bls.transit_mask(time, best_period, best_duration, best_t0)
    if transit_mask.sum() == 0 or (~transit_mask).sum() == 0:
        raise ValueError("BLS transit mask is empty or degenerate.")

    phase, folded_flux = fold_arrays(time, flux, best_period, best_t0)
    phase01, folded_flux01 = phase01_arrays(time, flux, best_period, best_t0)

    _, binned = binned_phase_curve(phase, folded_flux, n_bins=PHASE_BINS)
    baseline = np.nanmedian(binned)
    minimum = np.nanmin(binned)
    phase_curve_depth = baseline - minimum
    phase_curve_scatter = np.nanstd(binned)

    primary_depth = event_depth_from_phase01(phase01, folded_flux01, center=0.0, half_width=0.03)
    secondary_depth = event_depth_from_phase01(phase01, folded_flux01, center=0.5, half_width=0.03)
    odd_even_diff, n_measured_transits = compute_odd_even_depth_difference(
        time, flux, best_period, best_duration, best_t0
    )
    symmetry_metric = compute_phase_symmetry_metric(phase, folded_flux)
    transit_sharpness = compute_transit_sharpness(phase, folded_flux)

    in_flux = flux[transit_mask]
    out_flux = flux[~transit_mask]

    cdpp_3h_ppm = float(flattened_lc.estimate_cdpp(transit_duration=6))
    cdpp_6p5h_ppm = float(flattened_lc.estimate_cdpp(transit_duration=13))
    cdpp_12h_ppm = float(flattened_lc.estimate_cdpp(transit_duration=24))

    power_median = float(np.nanmedian(powers))
    power_std = float(np.nanstd(powers))
    power_zscore = np.nan if power_std == 0 else (best_power - power_median) / power_std

    best_depth_ppm = best_depth * 1e6
    primary_depth_ppm = np.nan if not np.isfinite(primary_depth) else primary_depth * 1e6
    secondary_depth_ppm = np.nan if not np.isfinite(secondary_depth) else secondary_depth * 1e6

    feature_row = {
        "n_points": len(time),
        "time_baseline_days": float(time.max() - time.min()),
        "flux_std": float(np.nanstd(flux)),
        "flux_mad": float(mad_normal(flux)),
        "cdpp_3h_ppm": cdpp_3h_ppm,
        "cdpp_6p5h_ppm": cdpp_6p5h_ppm,
        "cdpp_12h_ppm": cdpp_12h_ppm,
        "bls_best_period": best_period,
        "bls_best_duration_hours": best_duration * 24.0,
        "bls_best_depth": best_depth,
        "bls_best_depth_ppm": best_depth_ppm,
        "bls_best_power": best_power,
        "bls_best_depth_snr": best_depth_snr,
        "bls_power_zscore": power_zscore,
        "duty_cycle": best_duration / best_period,
        "estimated_transit_count": (time.max() - time.min()) / best_period,
        "in_transit_fraction": float(transit_mask.mean()),
        "transit_contrast": float(np.nanmedian(out_flux) - np.nanmedian(in_flux)),
        "phase_curve_depth": float(phase_curve_depth),
        "phase_curve_scatter": float(phase_curve_scatter),
        "primary_depth_ppm": primary_depth_ppm,
        "secondary_depth_ppm": secondary_depth_ppm,
        "secondary_to_primary_ratio": np.nan if not np.isfinite(primary_depth_ppm) or primary_depth_ppm == 0 else secondary_depth_ppm / primary_depth_ppm,
        "odd_even_depth_difference": float(odd_even_diff) if np.isfinite(odd_even_diff) else np.nan,
        "n_measured_transits": int(n_measured_transits),
        "phase_symmetry_metric": float(symmetry_metric) if np.isfinite(symmetry_metric) else np.nan,
        "transit_sharpness": float(transit_sharpness) if np.isfinite(transit_sharpness) else np.nan,
        "depth_over_cdpp_6p5h": np.nan if cdpp_6p5h_ppm == 0 else best_depth_ppm / cdpp_6p5h_ppm,
        "depth_over_flux_mad": np.nan if mad_normal(flux) == 0 else best_depth / mad_normal(flux),
    }

    if catalog_period is not None and np.isfinite(catalog_period):
        feature_row["catalog_period_days"] = float(catalog_period)
        feature_row["period_recovery_abs_error_days"] = abs(best_period - float(catalog_period))
        feature_row["period_recovery_relative_error"] = abs(best_period - float(catalog_period)) / float(catalog_period)

    return feature_row, results

## Building the feature table

This is where the notebook earns its keep.

For each target, I download and prepare the light curve, run the BLS search, extract the feature vector, and record any failures. I do not hide failures because failures are part of the real behavior of a pipeline. If a few targets fail because the archive product is missing, the download breaks, or the cleaned curve becomes unusably short, that is worth logging. A professional notebook does not quietly drop bad cases and pretend the workflow was perfectly smooth.

In [ ]:
def build_feature_table(targets: pd.DataFrame):
    records = []
    failures = []

    for row in targets.itertuples(index=False):
        try:
            _, flattened = download_and_prepare_lightcurve(row.kepid)
            features, _ = extract_transit_features(flattened, catalog_period=row.koi_period)

            record = {
                "kepid": int(row.kepid),
                "kepoi_name": row.kepoi_name,
                "koi_disposition": row.koi_disposition,
                "negative_label_source": row.negative_label_source,
                "label": int(row.label),
                "catalog_depth_ppm": row.koi_depth,
                "catalog_duration_hours": row.koi_duration,
                "catalog_period_days": row.koi_period,
            }
            record.update(features)
            records.append(record)
        except Exception as exc:
            failures.append({
                "kepid": int(row.kepid),
                "kepoi_name": row.kepoi_name,
                "koi_disposition": row.koi_disposition,
                "label": int(row.label),
                "error": str(exc),
            })

    return pd.DataFrame(records), pd.DataFrame(failures)

In [ ]:
if REBUILD_FEATURES or not FEATURE_CACHE_PATH.exists():
    features_df, failures_df = build_feature_table(targets_df)
    features_df.to_csv(FEATURE_CACHE_PATH, index=False)
    failures_df.to_csv(FAILURE_CACHE_PATH, index=False)
else:
    features_df = pd.read_csv(FEATURE_CACHE_PATH)
    failures_df = pd.read_csv(FAILURE_CACHE_PATH) if FAILURE_CACHE_PATH.exists() else pd.DataFrame()

print("Successful targets:", len(features_df))
print("Failed targets:", len(failures_df))
features_df.head()

## Sanity checks before modeling

Before I even think about fitting a classifier, I want to check three things.

I want to know whether the class balance survived the feature-extraction stage. If one class failed much more often than the other, that can bias the prototype.

I want to know where missing values exist. Some of the morphology features are allowed to be missing occasionally because they require enough phase coverage or enough detected events. That is acceptable. I can impute them later. But I want to see that pattern instead of stumbling over it by accident.

I also want to inspect the numerical ranges. If a feature that should be positive is wildly negative or if a recovery error is exploding for every target, that tells me the pipeline needs adjustment before the model gets anywhere near it.

In [ ]:
features_df.groupby(["label", "koi_disposition"]).size()

In [ ]:
features_df.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
preview_cols = [
    "bls_best_period", "bls_best_duration_hours", "bls_best_depth_ppm",
    "bls_best_depth_snr", "cdpp_6p5h_ppm", "depth_over_cdpp_6p5h",
    "secondary_depth_ppm", "odd_even_depth_difference", "period_recovery_relative_error"
]
features_df[preview_cols].describe().T

In [ ]:
if len(failures_df) > 0:
    failures_df.head(10)
else:
    print("No failures were logged in this run.")

## Early visual inspection of feature behavior

I do not want the first time I look at separation to be after the model has already produced a metric. That would make me too passive. I prefer to look at the features directly.

If the feature engineering is doing anything useful, I should expect at least some structure in distributions such as BLS depth SNR, depth relative to CDPP, odd-even difference, and secondary-event ratio. I am not expecting perfect separation. That would be suspicious in a small first prototype. I am looking for enough structure to justify training a classifier.

In [ ]:
plot_features = [
    "bls_best_depth_snr",
    "depth_over_cdpp_6p5h",
    "odd_even_depth_difference",
    "secondary_to_primary_ratio",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.ravel(), plot_features):
    for disposition, group in features_df.groupby("koi_disposition"):
        series = group[feature].replace([np.inf, -np.inf], np.nan).dropna()
        if len(series) > 0:
            ax.hist(series, bins=15, alpha=0.6, label=disposition)
    ax.set_title(feature)
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()

## Looking at actual targets, not only tables

One of the easiest ways to make a notebook feel fake is to reduce everything to feature tables and model scores. I do not want that. I want the notebook to keep returning to the underlying light curves.

For that reason I inspect example targets visually. For each example I show the stitched light curve, the BLS periodogram, and the phase-folded signal around the recovered transit. This keeps the model grounded in actual astrophysical behavior. It also helps me later when I need to speak about the project. I can point to a real signal and explain how it became a row in the dataset.

In [ ]:
def plot_target_diagnostics(kepid: int, title_prefix: str = ""):
    stitched, flattened = download_and_prepare_lightcurve(kepid)
    features, bls_results = extract_transit_features(flattened)

    time_raw = stitched.time.value
    flux_raw = np.asarray(getattr(stitched.flux, "value", stitched.flux), dtype=float)

    time = flattened.time.value
    flux = np.asarray(getattr(flattened.flux, "value", flattened.flux), dtype=float)

    periods = np.asarray(bls_results.period, dtype=float)
    powers = np.asarray(bls_results.power, dtype=float)
    best_idx = int(np.argmax(powers))

    best_period = float(np.asarray(bls_results.period)[best_idx])
    best_t0 = float(np.asarray(bls_results.transit_time)[best_idx])

    phase, folded_flux = fold_arrays(time, flux, best_period, best_t0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    axes[0].plot(time_raw, flux_raw, ".", ms=1, alpha=0.6)
    axes[0].set_title(f"{title_prefix}Stitched light curve")
    axes[0].set_xlabel("Time [BKJD]")
    axes[0].set_ylabel("Normalized flux")

    axes[1].plot(periods, powers, lw=1.5)
    axes[1].axvline(best_period, linestyle="--")
    axes[1].set_title("BLS periodogram")
    axes[1].set_xlabel("Period [days]")
    axes[1].set_ylabel("BLS power")

    axes[2].plot(phase, folded_flux, ".", ms=2, alpha=0.45)
    axes[2].set_xlim(-0.12, 0.12)
    axes[2].set_title("Phase-folded signal near primary event")
    axes[2].set_xlabel("Orbital phase")
    axes[2].set_ylabel("Flattened flux")

    plt.tight_layout()
    plt.show()

    return features

In [ ]:
example_confirmed = int(features_df.loc[features_df["label"] == 1, "kepid"].iloc[0])
example_negative = int(features_df.loc[features_df["label"] == 0, "kepid"].iloc[0])

print("Example confirmed KIC:", example_confirmed)
print("Example negative KIC:", example_negative)

In [ ]:
confirmed_example_features = plot_target_diagnostics(example_confirmed, title_prefix="Confirmed example: ")
confirmed_example_features

In [ ]:
negative_example_features = plot_target_diagnostics(example_negative, title_prefix="Negative example: ")
negative_example_features

## Choosing the model family

For a first prototype, I do not need a huge model. I need a model family that is fast, reasonably stable on small tabular data, and easy to explain.

I therefore start with two models.

The first is logistic regression. I like it because it is a disciplined baseline. If logistic regression performs above chance, that tells me the feature engineering itself already contains usable signal. It is also easy to explain because the model combines weighted evidence across features.

The second is a random forest. I add it because the relationship between transit features is not necessarily linear. A random forest can model non-linear splits and feature interactions without forcing me to commit to a more opaque or overcomplicated first model. It also gives me a straightforward feature-importance view afterward.

I am not treating the classifier as the whole project. The classifier sits on top of the signal-extraction work. That order matters. A model can only be as coherent as the data representation I give it.

In [ ]:
model_features = [
    "n_points",
    "time_baseline_days",
    "flux_std",
    "flux_mad",
    "cdpp_3h_ppm",
    "cdpp_6p5h_ppm",
    "cdpp_12h_ppm",
    "bls_best_period",
    "bls_best_duration_hours",
    "bls_best_depth_ppm",
    "bls_best_power",
    "bls_best_depth_snr",
    "bls_power_zscore",
    "duty_cycle",
    "estimated_transit_count",
    "in_transit_fraction",
    "transit_contrast",
    "phase_curve_depth",
    "phase_curve_scatter",
    "primary_depth_ppm",
    "secondary_depth_ppm",
    "secondary_to_primary_ratio",
    "odd_even_depth_difference",
    "n_measured_transits",
    "phase_symmetry_metric",
    "transit_sharpness",
    "depth_over_cdpp_6p5h",
    "depth_over_flux_mad",
    "period_recovery_relative_error",
]

X = features_df[model_features].copy()
y = features_df["label"].copy()

X_train, X_test, y_train, y_test, train_meta, test_meta = train_test_split(
    X,
    y,
    features_df[["kepid", "kepoi_name", "koi_disposition"]],
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

logreg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))
])

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

models = {
    "Logistic Regression": logreg_pipeline,
    "Random Forest": rf_pipeline,
}

## Why I evaluate with cross-validation before the holdout result

This dataset is not huge. That means a single train-test split can be lucky or unlucky. If I report only one split, I risk confusing a split-specific result with a true model pattern.

Cross-validation helps me reduce that risk. It repeatedly reshuffles the train-validation partition in a stratified way and measures how stable the model looks across folds. I still keep a final holdout set afterward, because that is useful for presentation and case-level error analysis. But I want the holdout result to come after I have already seen whether the model is at least somewhat stable.

This is one of those small details that makes a notebook feel more professional. It shows that I am not hunting for one flattering number and stopping the moment I get it.

In [ ]:
cv_splits = min(5, y.value_counts().min())
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_rows = []

for model_name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    row = {"model": model_name}
    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = np.mean(values)
        row[f"{metric_name}_std"] = np.std(values)
    cv_rows.append(row)

cv_results_df = pd.DataFrame(cv_rows).sort_values("roc_auc_mean", ascending=False)
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)
cv_results_df

## Selecting the stronger prototype model and fitting it

The model with the better cross-validation profile becomes my leading prototype model. I am not claiming that it is the final winner forever. I am simply choosing the model that, in this notebook run, gives the better balance of discrimination and stability.

After that, I fit the selected model on the training split and evaluate it on the holdout split. This gives me the two levels of evidence I want: cross-validated behavior across the full sample and concrete predictions on unseen held-out cases.

In [ ]:
best_model_name = cv_results_df.iloc[0]["model"]
best_pipeline = models[best_model_name]

print("Selected prototype model:", best_model_name)

best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Negative", "Confirmed"]))
print(f"Holdout ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Confirmed"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f"{best_model_name} confusion matrix")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title(f"{best_model_name} ROC curve")

plt.tight_layout()
plt.show()

## Reading the metrics without bluffing

This is where I need to stay disciplined. Metrics can look persuasive very quickly, but I do not want to oversell a first prototype.

If the scores are clearly above chance, that is already useful evidence. It means the pipeline has learned some repeatable structure from real light curves and not just from archive bookkeeping. That is a meaningful result for this stage of the project.

At the same time, the sample is small, the feature family is still first-generation, and the negative class design, while stronger than a naive approach, is still not the same thing as a final mission-grade vetting benchmark. So the right interpretation is not “the problem is solved.” The right interpretation is “the workflow is alive, coherent, and already capable of non-trivial discrimination.”

## Error analysis: the part that makes the prototype feel real

This section matters a lot to me.

A notebook becomes much more professional the moment it stops at its own mistakes and studies them. A confusion matrix is useful, but it is still abstract. I want to inspect real cases that became true positives, true negatives, false positives, and false negatives.

That lets me answer better questions. What does the model get confidently right? What kinds of non-planetary cases still look very planet-like? Which confirmed cases confuse the model? Are the mistakes tied to weak depth, noisy light curves, secondary events, or poor period recovery?

This is also the section I would use heavily in a presentation because it proves that I understand the model as a behavior, not only as a score.

In [ ]:
prediction_df = test_meta.copy().reset_index(drop=True)
prediction_df["actual"] = y_test.reset_index(drop=True)
prediction_df["predicted"] = y_pred
prediction_df["proba_confirmed"] = y_proba
prediction_df["case_type"] = np.select(
    [
        (prediction_df["actual"] == 1) & (prediction_df["predicted"] == 1),
        (prediction_df["actual"] == 0) & (prediction_df["predicted"] == 0),
        (prediction_df["actual"] == 0) & (prediction_df["predicted"] == 1),
        (prediction_df["actual"] == 1) & (prediction_df["predicted"] == 0),
    ],
    ["True Positive", "True Negative", "False Positive", "False Negative"],
    default="Other"
)

prediction_df.to_csv(PREDICTIONS_PATH, index=False)
prediction_df.sort_values("proba_confirmed", ascending=False)

In [ ]:
prediction_df["case_type"].value_counts()

In [ ]:
def select_representative_cases(pred_df: pd.DataFrame):
    selections = {}

    tp = pred_df[pred_df["case_type"] == "True Positive"].sort_values("proba_confirmed", ascending=False)
    tn = pred_df[pred_df["case_type"] == "True Negative"].sort_values("proba_confirmed", ascending=True)
    fp = pred_df[pred_df["case_type"] == "False Positive"].sort_values("proba_confirmed", ascending=False)
    fn = pred_df[pred_df["case_type"] == "False Negative"].sort_values("proba_confirmed", ascending=True)

    if len(tp) > 0:
        selections["True Positive"] = tp.iloc[0]
    if len(tn) > 0:
        selections["True Negative"] = tn.iloc[0]
    if len(fp) > 0:
        selections["False Positive"] = fp.iloc[0]
    if len(fn) > 0:
        selections["False Negative"] = fn.iloc[0]

    return selections

representative_cases = select_representative_cases(prediction_df)
list(representative_cases.keys())

In [ ]:
for case_name, case_row in representative_cases.items():
    print("=" * 80)
    print(case_name)
    print(case_row[["kepid", "kepoi_name", "koi_disposition", "actual", "predicted", "proba_confirmed"]])

In [ ]:
for case_name, case_row in representative_cases.items():
    title = (
        f"{case_name} | KIC {int(case_row['kepid'])} | "
        f"actual={int(case_row['actual'])} | predicted={int(case_row['predicted'])} | "
        f"P(confirmed)={case_row['proba_confirmed']:.3f}"
    )
    _ = plot_target_diagnostics(int(case_row["kepid"]), title_prefix=title + " | ")

## Interpreting the mistake patterns

When I look at false positives in this context, I need to remember that the negative class is not composed of empty stars with no transit-like behavior. Many negative cases can still contain strong repeating dips. That is precisely why they are interesting. Some will be eclipsing binaries, some may show secondary events, some may have odd-even differences, and some may simply look convincing in a small feature space.

False negatives, on the other hand, are especially useful because they tell me where confirmed planets still fail to look sufficiently planet-like to the current feature set and model. A weak signal, noisy baseline, poor phase coverage, or a recovered period that is slightly off can all make a confirmed case look less compelling in the engineered features than it really is.

That is why this section is not an afterthought. It tells me what the next version of the project should improve.

## Interpreting the model itself

A model score tells me how well the model discriminates. Feature importance tells me what kind of evidence it relies on most.

If the top features include things like BLS depth SNR, depth relative to CDPP, odd-even difference, secondary-event behavior, or period-recovery quality, that is reassuring. Those are all physically or procedurally meaningful signals in a transit-vetting problem. In other words, the classifier would not only be accurate to some degree. It would also be accurate for reasons that make sense in the astronomy context.

In [ ]:
if best_model_name == "Random Forest":
    fitted_model = best_pipeline.named_steps["model"]
    importance_df = pd.DataFrame({
        "feature": model_features,
        "importance": fitted_model.feature_importances_
    }).sort_values("importance", ascending=False)
else:
    fitted_model = best_pipeline.named_steps["model"]
    importance_df = pd.DataFrame({
        "feature": model_features,
        "importance": fitted_model.coef_[0]
    }).sort_values("importance", key=np.abs, ascending=False)

importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
importance_df.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
top_imp = importance_df.head(15).iloc[::-1]
ax.barh(top_imp["feature"], top_imp["importance"])
ax.set_title(f"Top feature importance view for {best_model_name}")
ax.set_xlabel("Importance" if best_model_name == "Random Forest" else "Coefficient")
plt.tight_layout()
plt.show()

## Optional interactive notebook moment

If I want to add a live demonstration moment inside Jupyter Lab, Lightkurve also supports interactive BLS inspection. I am leaving the cell below commented because it depends on the frontend environment handling the widget stack cleanly, but it is useful as a presentation option.

In [ ]:
# Optional interactive view in Jupyter Lab:
# stitched, flattened = download_and_prepare_lightcurve(example_confirmed)
# flattened.interact_bls()

## What I would say this prototype achieves

This notebook gives me a defensible first version of the project.

I start with real Kepler archive objects and a label strategy that tries to use a stronger negative source where possible. I retrieve the actual mission light curves. I clean, normalize, stitch, and flatten them. I search for transit-like signals using Box Least Squares, which is a standard tool for this kind of problem. I summarize the recovered candidate using engineered features that reflect signal strength, noise level, event morphology, and consistency with the archive period. Then I train and evaluate a first tabular classifier on top of that representation.

That is enough to show a serious prototype. It is not only a visualization notebook. It is not only a model notebook. It is an actual pipeline from archive object to machine-learning decision.

## Limitations I would state openly

I would not hide the limits of this notebook because the limits are part of what makes the next version obvious.

The sample is still small, so the reported metrics are useful but not final. The negative class, while stronger than a naive choice, is still not identical to a fully mission-grade evaluation design. The BLS search grid is intentionally compact to keep the notebook demonstrable. The feature family is much better than a bare-baseline version, but it is still first-generation feature engineering.

Those limits do not make the prototype weak. They simply locate it in the project timeline. This is a first version designed to prove that the pipeline works and that the next stage is worth building.

## Where I would take the project next

The next version would strengthen each stage instead of replacing the whole workflow.

I would grow the sample size carefully and preserve the stronger label strategy. I would add richer vetting features, especially features that separate eclipsing binaries and instrumental artifacts more clearly from planetary transits. I would examine whether multiple candidate peaks per light curve carry useful information instead of keeping only the strongest BLS solution. I would deepen the error analysis and track specific failure modes across runs. After that, and only after that, I would compare this engineered-feature approach against models that ingest a more direct representation of the folded light curve.

That path makes sense because the current notebook already gives me something stable to build on. I do not need to restart. I need to refine.

## Short study summary for myself

If I need to explain this notebook from memory, I can summarize the story like this.

I am not pretending to detect exoplanets from raw stars. I am building a transit candidate vetting prototype. I use real Kepler archive objects, retrieve their light curves with Lightkurve, clean and flatten the data, and use Box Least Squares to recover the strongest periodic box-shaped dip. From that recovered signal I build a feature vector that describes the candidate’s period, duration, depth, significance, noise environment, and transit shape. I then train a classifier to separate confirmed cases from strong negative cases. Finally, I do not stop at the metric. I inspect mistakes on real light curves to understand what the model is seeing and where the pipeline still needs work.

That is the backbone of the prototype. If I can explain those sentences clearly, then I understand the notebook well enough to present it.